# 07 · Híbrido SMOTE-NC → CTGAN: selección de hiperparámetros

Algoritmo 1 del artículo:
1. SMOTE-NC amplía la minoría hasta el piso α·M (α = 0.5, fijado a priori).
2. La minoría ampliada vuelve a la escala original.
3. CTGAN se entrena solo con esa minoría ampliada.
4. Se muestrean M − m registros de CTGAN; los registros interpolados por SMOTE-NC se descartan.

Aquí se eligen los hiperparámetros de la etapa CTGAN por el AUC-PR en validación del proxy. En esta búsqueda, SMOTE-NC usa sus parámetros por defecto (k = 5); el protocolo final (`10_multirun_fair.py`) usa el k seleccionado en el notebook 04.

In [ ]:
import json
import math
import random
import joblib
import numpy as np
import pandas as pd
import torch
from collections import Counter
from ctgan import CTGAN
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)
from utils.config_loader import load_config

config = load_config("dataset_config.yaml")
RANDOM_STATE = config.get("random_state", 42)
print("Dataset:", config["dataset_name"])
USE_GPU = torch.cuda.is_available()
ALPHA_SMOTE = 0.5
scaler = joblib.load("outputs/data/scaler.pkl")
categorical_cols = config.get("categorical_features", [])
PARAM_GRID = [
    {"epochs": 300, "batch_size": 500, "pac": 10},
    {"epochs": 500, "batch_size": 500, "pac": 10},
    {"epochs": 500, "batch_size": 1000, "pac": 10},
]


def set_seed(seed):
    """Fija las semillas de Python, NumPy y PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Datos

In [ ]:
SPLIT_DIR = "outputs/splits"

X_train_raw = pd.read_csv(f"{SPLIT_DIR}/X_train_raw_in.csv")      # entrenamiento interno, escala original
X_train_proc = pd.read_csv(f"{SPLIT_DIR}/X_train_in.csv")         # entrenamiento interno, escalado
y_train = pd.read_csv(f"{SPLIT_DIR}/y_train_in.csv")["y"].values
X_val_proc = pd.read_csv(f"{SPLIT_DIR}/X_val_in.csv")             # validación, escalada
y_val = pd.read_csv(f"{SPLIT_DIR}/y_val_in.csv")["y"].values

counter = Counter(y_train)
minority = min(counter, key=counter.get)
majority = max(counter, key=counter.get)
n_to_generate = counter[majority] - counter[minority]
print(f"Distribución del entrenamiento interno: { {int(k): int(v) for k, v in counter.items()} }")
print(f"Registros sintéticos a generar (M - m): {n_to_generate}")
M, m = counter[majority], counter[minority]
num_cols = [c for c in X_train_proc.columns if c not in categorical_cols]

## Pasos 1 y 2: piso SMOTE-NC y vuelta a la escala original

In [ ]:
floor = int(math.ceil(ALPHA_SMOTE * M))
cat_idx = [X_train_proc.columns.get_loc(c) for c in categorical_cols]
if cat_idx:
    sampler = SMOTENC(categorical_features=cat_idx, sampling_strategy={minority: floor}, random_state=RANDOM_STATE)
else:
    sampler = SMOTE(sampling_strategy={minority: floor}, random_state=RANDOM_STATE)
X_sm_proc, y_sm = sampler.fit_resample(X_train_proc, y_train)

X_sm_raw = X_sm_proc.copy()
X_sm_raw[num_cols] = scaler.inverse_transform(X_sm_proc[num_cols])

# Minoría ampliada (real + interpolada). La columna "target" es constante e identifica la clase.
train_ctgan = X_sm_raw[y_sm == minority].copy()
train_ctgan["target"] = minority
print(f"Minoría ampliada para entrenar CTGAN: {len(train_ctgan)} registros (piso = {floor})")

## Pasos 3 y 4: CTGAN sobre la minoría ampliada y búsqueda de hiperparámetros

In [ ]:
def sample_minority(model, n_needed, max_attempts=15):
    """Muestrea de CTGAN hasta reunir n_needed registros de la clase minoritaria."""
    parts, attempts = [], 0
    while sum(len(p) for p in parts) < n_needed and attempts < max_attempts:
        synth = model.sample(n_needed * 5)
        parts.append(synth[synth["target"] == minority])
        attempts += 1
    synth_all = pd.concat(parts, ignore_index=True)
    if len(synth_all) > n_needed:
        synth_all = synth_all.sample(n=n_needed, random_state=RANDOM_STATE)
    return synth_all.drop(columns=["target"])


def proxy_auc_pr(synth_raw):
    """AUC-PR en validación de una regresión logística entrenada con datos reales + sintéticos."""
    synth_proc = synth_raw.copy()
    num_cols = [c for c in synth_raw.columns if c not in categorical_cols]
    synth_proc[num_cols] = scaler.transform(synth_raw[num_cols].astype(float))
    X_proxy = pd.concat([X_train_proc, synth_proc], ignore_index=True)
    y_proxy = np.concatenate([y_train, np.full(len(synth_proc), minority)])
    proxy = LogisticRegression(max_iter=3000)
    proxy.fit(X_proxy, y_proxy)
    return average_precision_score(y_val, proxy.predict_proba(X_val_proc)[:, 1])


best_score, best_params, best_synth = -np.inf, None, None
for params in PARAM_GRID:
    set_seed(RANDOM_STATE)
    model = CTGAN(epochs=params["epochs"], batch_size=params["batch_size"], pac=params["pac"],
                  enable_gpu=USE_GPU)
    model.fit(train_ctgan, categorical_cols + ["target"])
    synth_raw = sample_minority(model, n_to_generate)
    if len(synth_raw) < n_to_generate:
        continue

    auc_pr = proxy_auc_pr(synth_raw)
    print(f"{params}: AUC-PR validación = {auc_pr:.4f}")
    if auc_pr > best_score:
        best_score, best_params, best_synth = auc_pr, params, synth_raw.copy()

## Guardado (registros reales + registros de CTGAN)

In [ ]:
OUTPUT_DIR = "outputs/hybrid"
os.makedirs(OUTPUT_DIR, exist_ok=True)
X_final = pd.concat([X_train_raw, best_synth], ignore_index=True)
y_final = np.concatenate([y_train, np.full(len(best_synth), minority)])
X_final.to_csv(f"{OUTPUT_DIR}/X_train_hybrid.csv", index=False)
pd.DataFrame({"y": y_final}).to_csv(f"{OUTPUT_DIR}/y_train_hybrid.csv", index=False)

with open(f"{OUTPUT_DIR}/best_config.json", "w") as f:
    json.dump({"best_auc_pr": float(best_score), "best_params": best_params,
               "alpha_smote": ALPHA_SMOTE}, f, indent=4)
print(f"Configuración seleccionada: {best_params} (AUC-PR = {best_score:.4f})")